# 03: Model Training

This notebook trains a fraud detection model using the processed features.

## Setup

In [0]:

%pip install catboost shap imbalanced-learn mlflow

import os, sys, json, pickle
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")          
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.catboost
import shap

# Add src to path
sys.path.insert(0, '/Workspace/Users/<user>/fraud_Detection_ML/src/model')

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window

from train import train_model
from evaluate import evaluate_model
from threshold  import find_best_threshold
from shap_explainer import compute_shap_reasons


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
spark = SparkSession.builder \
    .appName("FraudDetectionModelTraining") \
    .config("spark.sql.shuffle.partitions", "16") \
    .getOrCreate()

print(f"Spark version: {spark.version}")


Spark version: 4.1.0


## Load Processed Data

In [0]:
# Load from Delta table written by notebook 02
df = spark.table("fraud_features")
print(f"Processed data loaded: {df.count():,} records")
df.printSchema()


Processed data loaded: 495,766 records
root
 |-- ssn: string (nullable = true)
 |-- cc_num: string (nullable = true)
 |-- first: string (nullable = true)
 |-- last: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- street: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- city_pop: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- acct_num: string (nullable = true)
 |-- profile: string (nullable = true)
 |-- trans_num: string (nullable = true)
 |-- trans_date: string (nullable = true)
 |-- trans_time: string (nullable = true)
 |-- unix_time: long (nullable = true)
 |-- category: string (nullable = true)
 |-- amt: double (nullable = true)
 |-- is_fraud: integer (nullable = true)
 |-- merchant: string (nullable = true)
 |-- merch_lat: double (nullable = true)
 |--

## Inspect Class Imbalance 

In [0]:
label_dist = df.groupBy("is_fraud").count().toPandas()
total = label_dist["count"].sum()
label_dist["pct"] = (label_dist["count"] / total * 100).round(2)
print("Class distribution:")
print(label_dist.to_string(index=False))

fraud_count   = int(label_dist.loc[label_dist.is_fraud == 1, "count"])
legit_count   = int(label_dist.loc[label_dist.is_fraud == 0, "count"])
imbalance_ratio = round(legit_count / fraud_count, 1)
print(f"\nImbalance ratio (legit:fraud) = {imbalance_ratio}:1")


Class distribution:
 is_fraud  count   pct
        0 487800 98.39
        1   7966  1.61

Imbalance ratio (legit:fraud) = 61.2:1


/home/spark-3df62cfa-b91e-469e-bb0e-04/.ipykernel/306/command-7125838881788027-2117377312:7: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  fraud_count   = int(label_dist.loc[label_dist.is_fraud == 1, "count"])
/home/spark-3df62cfa-b91e-469e-bb0e-04/.ipykernel/306/command-7125838881788027-2117377312:8: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  legit_count   = int(label_dist.loc[label_dist.is_fraud == 0, "count"])


## Train-Test Split (Time-Based)

In [0]:
# Sort by unix_time (temporal ordering – prevents data leakage)
df = df.orderBy("unix_time")

total_count = df.count()
train_end   = int(total_count * 0.70)
val_end     = int(total_count * 0.85)

window = Window.orderBy("unix_time")
df = df.withColumn("row_idx", F.row_number().over(window))

train_df = df.filter(F.col("row_idx") <= train_end)
val_df   = df.filter((F.col("row_idx") > train_end) & (F.col("row_idx") <= val_end))
test_df  = df.filter(F.col("row_idx") > val_end)

print(f"Train : {train_df.count():,}")
print(f"Val   : {val_df.count():,}")
print(f"Test  : {test_df.count():,}")


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Train : 347,036
Val   : 74,365
Test  : 74,365


## Prepare Features and Labels

In [0]:
# All features that come out of the feature-engineering notebooks
FEATURE_COLS = [
    # ── Numeric / Static ──────────────────────────────────────────
    "amt",                    # transaction amount
    "city_pop",               # customer city population
    "hour",                   # hour of day (0-23)
    "day_of_week",            # day of week (1=Sunday)
    "month",                  # calendar month
    # ── Geospatial ────────────────────────────────────────────────
    "distance",               # Euclidean customer-merchant distance
    "haversine_distance",     # Great-circle distance (km)  ← was unused
    # ── Window / Velocity features ────────────────────────────────
    "txn_count_1h",           # # txns by card in last 1 hour
    "txn_count_24h",          # # txns by card in last 24 hours
    "avg_amt_24h",            # avg spend in last 24 hours
    "spend_24h",              # total spend in last 24 hours
    "unique_merchants_24h",   # distinct merchants in last 24 hours
    "time_since_last_txn",    # seconds since previous txn on this card
    # ── Lookup / Fraud-Rate features ──────────────────────────────
    "category_fraud_rate",    # historical fraud rate per category
    "category_txn_count",     # # txns per category (volume signal)
    "merchant_fraud_rate",    # historical fraud rate per merchant
    "merchant_txn_count",     # # txns per merchant
    "state_fraud_rate",       # historical fraud rate per state
    # ── Categorical (passed as-is to CatBoost) ────────────────────
    "category", "merchant", "state", "gender", "city", "zip", "job",
]

# Keep only columns that actually exist in this run's processed table
feature_cols = [c for c in FEATURE_COLS if c in df.columns]
label_col    = "is_fraud"

print(f"Using {len(feature_cols)} features:")
for c in feature_cols:
    print(f"  {c}")


Using 20 features:
  amt
  city_pop
  hour
  day_of_week
  month
  distance
  haversine_distance
  txn_count_1h
  txn_count_24h
  avg_amt_24h
  spend_24h
  unique_merchants_24h
  time_since_last_txn
  category
  merchant
  state
  gender
  city
  zip
  job


In [0]:
X_train_pd = train_df.select(feature_cols).toPandas()
y_train_pd = train_df.select(label_col).toPandas()[label_col].astype(int)

X_val_pd   = val_df.select(feature_cols).toPandas()
y_val_pd   = val_df.select(label_col).toPandas()[label_col].astype(int)

X_test_pd  = test_df.select(feature_cols).toPandas()
y_test_pd  = test_df.select(label_col).toPandas()[label_col].astype(int)

print(f"X_train: {X_train_pd.shape}  Fraud: {y_train_pd.sum():,}")
print(f"X_val  : {X_val_pd.shape}    Fraud: {y_val_pd.sum():,}")
print(f"X_test : {X_test_pd.shape}   Fraud: {y_test_pd.sum():,}")


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data 

X_train: (347036, 20)  Fraud: 6,295
X_val  : (74365, 20)    Fraud: 1,090
X_test : (74365, 20)   Fraud: 581


## Class Balancing via Hybrid SMOTENC + Undersample

In [0]:
from imblearn.over_sampling import SMOTENC
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
import pandas as pd

# ── 1. Separate your column types ───────────────────────────────────────
CAT_NAMES = ["category", "merchant", "state", "gender", "city", "zip", "job"]
cat_cols_present = [c for c in CAT_NAMES if c in feature_cols]
numeric_cols = [c for c in feature_cols if c not in cat_cols_present]

# Calculate correct numerical indices based on feature list
cat_indices = [feature_cols.index(c) for c in cat_cols_present]

# ── 2. Diagnose and Clear NaNs (The Fix) ────────────────────────────────
print("Checking for hidden missing values...")
nans_per_col = X_train_pd.isna().sum()
cols_with_nans = nans_per_col[nans_per_col > 0]

if not cols_with_nans.empty:
    print(f"⚠️ Found missing values in these columns:\n{cols_with_nans}\n")
    print("Filling missing values to satisfy SMOTENC parameters...")
    
    # Fill categorical NaNs with a fallback string
    for c in cat_cols_present:
        X_train_pd[c] = X_train_pd[c].fillna("UNKNOWN")
        
    # Fill numeric NaNs (like lookup rates) with 0.0 or column median
    for c in numeric_cols:
        # Filling fraud rate lookups with 0 makes mathematical sense (assume clean if brand new)
        if "fraud_rate" in c:
            X_train_pd[c] = X_train_pd[c].fillna(0.0)
        else:
            X_train_pd[c] = X_train_pd[c].fillna(X_train_pd[c].median())
else:
    print("✅ No NaNs found in raw feature array.")

# ── 3. Run Your Resampling Pipeline ────────────────────────────────────
print(f"\nCategorical indices mapped: {cat_indices}")
print(f"Train shape before resampling: {X_train_pd.shape}")
print(f"Fraud count before: {y_train_pd.sum():,} ({y_train_pd.mean()*100:.2f}%)")

over = SMOTENC(
    categorical_features=cat_indices,
    sampling_strategy=0.10,   # raise minority to 10% of majority
    random_state=42
)
under = RandomUnderSampler(
    sampling_strategy=0.33,  # final ratio ~1:3
    random_state=42
)

pipeline = Pipeline(steps=[("over", over), ("under", under)])

# This will now execute perfectly without the ValueError!
X_res, y_res = pipeline.fit_resample(X_train_pd, y_train_pd)

print(f"\nTrain shape after resampling : {X_res.shape}")
print(f"Fraud count after : {y_res.sum():,} ({y_res.mean()*100:.2f}%)")

Checking for hidden missing values...
⚠️ Found missing values in these columns:
time_since_last_txn    886
dtype: int64

Filling missing values to satisfy SMOTENC parameters...

Categorical indices mapped: [13, 14, 15, 16, 17, 18, 19]
Train shape before resampling: (347036, 20)
Fraud count before: 6,295 (1.81%)

Train shape after resampling : (137328, 20)
Fraud count after : 34,074 (24.81%)


## Testing the Model

In [0]:
# compare balance resampled and imbalanced ratio (result : imbalanced better)

print("Configuring fast evaluation arrays...")
# Using a 20% sample of the resampled data just to run this comparison fast
sample_size = int(len(X_res) * 0.20)
np.random.seed(42)
shuffle_idx = np.random.permutation(len(X_res))[:sample_size]

X_train_comp = X_res.iloc[shuffle_idx] if isinstance(X_res, pd.DataFrame) else X_res[shuffle_idx]
y_train_comp = y_res.iloc[shuffle_idx] if isinstance(y_res, pd.Series) else y_res[shuffle_idx]

# Keep your validation set pure
X_val_comp = X_val_pd
y_val_comp = y_val_pd

# Define structural configurations
cat_features_for_cb = [c for c in CAT_NAMES if c in feature_cols]
base_params = {
    "iterations": 300,  # low iterations just for quick testing
    "learning_rate": 0.05,
    "depth": 6,
    "eval_metric": "AUC",
    "random_seed": 42,
    "verbose": False,
    "cat_features": cat_features_for_cb
}

results = {}

# ── 2. Run Trial 1: The Double-Counting Penalty (61.2) ────────────────────────
print("\n🏃 Running Trial 1: Using Raw Imbalance Ratio (61.2)...")
params_trial1 = base_params.copy()
params_trial1["class_weights"] = [1, 61.2]

model_t1 = CatBoostClassifier(**params_trial1)
model_t1.fit(X_train_comp, y_train_comp, eval_set=(X_val_comp, y_val_comp))

# Evaluate at standard 0.5 threshold for raw baseline comparison
preds_t1 = model_t1.predict(X_val_comp)
proba_t1 = model_t1.predict_proba(X_val_comp)[:, 1]

results["Raw Ratio (61.2)"] = {
    "AUC": roc_auc_score(y_val_comp, proba_t1),
    "Precision": precision_score(y_val_comp, preds_t1, zero_division=0),
    "Recall": recall_score(y_val_comp, preds_t1),
    "F1": f1_score(y_val_comp, preds_t1, zero_division=0)
}

# ── 3. Run Trial 2: The Resampled Balanced Penalty (3.0) ──────────────────────
print("🏃 Running Trial 2: Using Balanced Resampled Ratio (3.0)...")
params_trial2 = base_params.copy()
params_trial2["class_weights"] = [1, 3.0]

model_t2 = CatBoostClassifier(**params_trial2)
model_t2.fit(X_train_comp, y_train_comp, eval_set=(X_val_comp, y_val_comp))

preds_t2 = model_t2.predict(X_val_comp)
proba_t2 = model_t2.predict_proba(X_val_comp)[:, 1]

results["Balanced Ratio (3.0)"] = {
    "AUC": roc_auc_score(y_val_comp, proba_t2),
    "Precision": precision_score(y_val_comp, preds_t2, zero_division=0),
    "Recall": recall_score(y_val_comp, preds_t2),
    "F1": f1_score(y_val_comp, preds_t2, zero_division=0)
}

# ── 4. Print the Comparative Matrix ──────────────────────────────────────────
print("\n" + "="*60)
print(f"{'WEIGHT STRATEGY':<22} | {'AUC':<6} | {'PRECISION':<9} | {'RECALL':<6} | {'F1-SCORE':<8}")
print("="*60)
for strategy, metrics in results.items():
    print(f"{strategy:<22} | {metrics['AUC']:.4f} | {metrics['Precision']:.4f}    | {metrics['Recall']:.4f} | {metrics['F1']:.4f}")
print("="*60)

Configuring fast evaluation arrays...

🏃 Running Trial 1: Using Raw Imbalance Ratio (61.2)...
🏃 Running Trial 2: Using Balanced Resampled Ratio (3.0)...

WEIGHT STRATEGY        | AUC    | PRECISION | RECALL | F1-SCORE
Raw Ratio (61.2)       | 0.9041 | 0.0179    | 0.9927 | 0.0351
Balanced Ratio (3.0)   | 0.4612 | 0.0163    | 0.9202 | 0.0321


In [0]:
# dropped SMOTE and tested scale_pos_weight = 12.0 (without balancing data is better)
# Metric comparisons (Native Scale Weight + High-Band Threshold Tuning)

print("Preparing sandboxed verification splits...")

# Using a 30% sample of the training data to ensure fast execution speeds
np.random.seed(42)
train_sample_size = int(len(X_train_pd) * 0.30)
shuffle_idx = np.random.permutation(len(X_train_pd))[:train_sample_size]

X_train_exp = X_train_pd.iloc[shuffle_idx].copy()
y_train_exp = y_train_pd.iloc[shuffle_idx].copy()

# Fill the single missing feature array locally to protect CatBoost metrics
X_train_exp["time_since_last_txn"] = X_train_exp["time_since_last_txn"].fillna(X_train_exp["time_since_last_txn"].median())
X_val_clean = X_val_pd.copy()
X_val_clean["time_since_last_txn"] = X_val_clean["time_since_last_txn"].fillna(X_train_exp["time_since_last_txn"].median())

# Calculate dynamic scale parameters natively
raw_imbalance_ratio = (len(y_train_exp) - y_train_exp.sum()) / y_train_exp.sum()
cat_features_for_cb = [c for c in CAT_NAMES if c in feature_cols]

# Core shared parameters for evaluation speed
base_params = {
    "iterations": 400, 
    "learning_rate": 0.05,
    "depth": 6,
    "eval_metric": "AUC",
    "random_seed": 42,
    "verbose": False,
    "cat_features": cat_features_for_cb
}

experimental_results = {}

# ── HELPER FUNCTION: Optimize threshold via validation loop ───────────────
def evaluate_strategy_with_tuning(model_obj, low_bound, high_bound):
    model_obj.fit(X_train_exp, y_train_exp, eval_set=(X_val_clean, y_val_pd))
    probs = model_obj.predict_proba(X_val_clean)[:, 1]
    
    best_thresh = 0.5
    best_f1 = 0.0
    
    # Search the assigned probability space for the best F1 score
    for thresh in np.linspace(low_bound, high_bound, 400):
        preds = (probs >= thresh).astype(int)
        score = f1_score(y_val_pd, preds, zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_thresh = thresh
            
    final_preds = (probs >= best_thresh).astype(int)
    return {
        "AUC": roc_auc_score(y_val_pd, probs),
        "Precision": precision_score(y_val_pd, final_preds, zero_division=0),
        "Recall": recall_score(y_val_pd, final_preds),
        "F1": f1_score(y_val_pd, final_preds, zero_division=0),
        "Threshold": best_thresh
    }

# ── RUNNING TRIAL 1 ───────────────────────────────────────────────────────
print(f"🏃 Evaluating Option 1: Native Scale Weight ({raw_imbalance_ratio:.2f}) + High-Band Tuning...")
params_o1 = base_params.copy()
params_o1["scale_pos_weight"] = raw_imbalance_ratio
experimental_results["Option 1 (Full Scale)"] = evaluate_strategy_with_tuning(
    CatBoostClassifier(**params_o1), low_bound=0.5, high_bound=0.999
)

# ── RUNNING TRIAL 2 ───────────────────────────────────────────────────────
print("🏃 Evaluating Option 2: Native Balanced Downscaling (3.0) + Mid-Band Tuning...")
params_o2 = base_params.copy()
params_o2["scale_pos_weight"] = 3.0
experimental_results["Option 2 (Scaled 3.0)"] = evaluate_strategy_with_tuning(
    CatBoostClassifier(**params_o2), low_bound=0.1, high_bound=0.95
)

# ── RUNNING TRIAL 3 ───────────────────────────────────────────────────────
print("🏃 Evaluating Option 3: Pure Natural Baseline (1.0) + Low-Band Tuning...")
params_o3 = base_params.copy()
params_o3["scale_pos_weight"] = 1.0
experimental_results["Option 3 (Natural Base)"] = evaluate_strategy_with_tuning(
    CatBoostClassifier(**params_o3), low_bound=0.001, high_bound=0.50
)

# ── 5. Display Comparative Summary Matrix ──────────────────────────────────
print("\n" + "="*85)
print(f"{'ARCHITECTURAL STRATEGY':<25} | {'AUC':<6} | {'PRECISION':<9} | {'RECALL':<6} | {'F1-SCORE':<8} | {'BEST THRESH':<10}")
print("="*85)
for strategy, metrics in experimental_results.items():
    print(f"{strategy:<25} | {metrics['AUC']:.4f} | {metrics['Precision']:.4f}    | {metrics['Recall']:.4f} | {metrics['F1']:.4f}   | {metrics['Threshold']:.4f}")
print("="*85)

Preparing sandboxed verification splits...
🏃 Evaluating Option 1: Native Scale Weight (54.26) + High-Band Tuning...
🏃 Evaluating Option 2: Native Balanced Downscaling (3.0) + Mid-Band Tuning...
🏃 Evaluating Option 3: Pure Natural Baseline (1.0) + Low-Band Tuning...

ARCHITECTURAL STRATEGY    | AUC    | PRECISION | RECALL | F1-SCORE | BEST THRESH
Option 1 (Full Scale)     | 0.8680 | 0.2562    | 0.5679 | 0.3531   | 0.9990
Option 2 (Scaled 3.0)     | 0.9167 | 0.0650    | 0.8761 | 0.1210   | 0.9500
Option 3 (Natural Base)   | 0.9278 | 0.0209    | 0.9422 | 0.0409   | 0.5000


In [0]:

# Raw Native Scale Weighting (Scale = 54.12) >> overfit data
# ── 1. Calculate the Option 1 Native Scale Weight ────────────────────
total_legit = len(y_train_pd) - y_train_pd.sum()
total_fraud = y_train_pd.sum()
raw_imbalance_ratio = total_legit / total_fraud

print("=" * 60)
print(f"📊 FULL TRAIN PROFILE | Total Legit: {total_legit:,} | Total Fraud: {total_fraud:,}")
print(f"🚀 OPTION 1 ACTIVE    | Native Scale Multiplier: {raw_imbalance_ratio:.4f}")
print("=" * 60)

# Map categorical names exactly as strings for CatBoost safety
cat_features_for_cb = [c for c in CAT_NAMES if c in feature_cols]

# ── 2. Configure Option 1 Hyper-parameters ───────────────────────────
model_params = {
    "iterations"           : 1500,
    "learning_rate"        : 0.03,      
    "depth"                : 7,
    "eval_metric"          : "AUC",
    "early_stopping_rounds": 100,
    "verbose"              : 200,
    "random_seed"          : 42,
    "loss_function"        : "Logloss",
    
    # Core Change: Applying full scale multiplier on pure, un-scrambled data
    "scale_pos_weight"     : raw_imbalance_ratio,
    
    "l2_leaf_reg"          : 5,
    "border_count"         : 128,
    "cat_features"         : cat_features_for_cb
}

# ── 3. Train the Model ───────────────────────────────────────────────
print("\n🏋️ Training full-scale CatBoost model...")
model = CatBoostClassifier(**model_params)
model.fit(
    X_train_pd, y_train_pd,
    eval_set=(X_val_pd, y_val_pd),
    use_best_model=True
)

# ── 4. High-Band Probability Threshold Tuning ─────────────────────────
print("\n🔍 Scanning high-band probability spectrum for the optimal F1 threshold...")
val_probs = model.predict_proba(X_val_pd)[:, 1]

best_threshold = 0.5
best_f1 = 0.0

# Aggressively scan the compressed high probability bands up to 0.9999
for thresh in np.linspace(0.9000, 0.9999, 1000):
    preds = (val_probs >= thresh).astype(int)
    score = f1_score(y_val_pd, preds, zero_division=0)
    if score > best_f1:
        best_f1 = score
        best_threshold = thresh

print(f"🎯 Optimal Cutoff Threshold Located: {best_threshold:.4f} (Validation F1: {best_f1:.4f})")

# ── 5. Evaluate on Held-Out Test Set ─────────────────────────────────
print("\n🏃 Executing final evaluation on unseen Test Split...")
test_probs = model.predict_proba(X_test_pd)[:, 1]
test_preds = (test_probs >= best_threshold).astype(int)

test_auc = roc_auc_score(y_test_pd, test_probs)
test_prec = precision_score(y_test_pd, test_preds, zero_division=0)
test_rec = recall_score(y_test_pd, test_preds)
test_f1 = f1_score(y_test_pd, test_preds, zero_division=0)

print("\n" + "=" * 50)
print(f"  VERIFIED PERFORMANCE METRICS (OPTION 1)")
print("=" * 50)
print(f"  Final Test AUC       : {test_auc:.4f}")
print(f"  Final Test Precision : {test_prec:.4f}")
print(f"  Final Test Recall    : {test_rec:.4f}")
print(f"  Final Test F1-Score  : {test_f1:.4f}")
print(f"  Operational Cutoff   : {best_threshold:.4f}")
print("=" * 50)

📊 FULL TRAIN PROFILE | Total Legit: 340,741 | Total Fraud: 6,295
🚀 OPTION 1 ACTIVE    | Native Scale Multiplier: 54.1288

🏋️ Training full-scale CatBoost model...
0:	test: 0.3299059	best: 0.3299059 (0)	total: 486ms	remaining: 12m 8s
200:	test: 0.7733776	best: 0.7835055 (179)	total: 1m 56s	remaining: 12m 32s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.7835054573
bestIteration = 179

Shrink model to first 180 iterations.

🔍 Scanning high-band probability spectrum for the optimal F1 threshold...
🎯 Optimal Cutoff Threshold Located: 0.9995 (Validation F1: 0.2708)

🏃 Executing final evaluation on unseen Test Split...

  VERIFIED PERFORMANCE METRICS (OPTION 1)
  Final Test AUC       : 0.7137
  Final Test Precision : 0.1225
  Final Test Recall    : 0.2565
  Final Test F1-Score  : 0.1658
  Operational Cutoff   : 0.9995


In [0]:
# Balanced Aggression Strategy (Scale = 12.0) >> better iteration

# Map categorical names exactly as strings for CatBoost safety
cat_features_for_cb = [c for c in CAT_NAMES if c in feature_cols]

# ── STRATEGY A CONFIGURATION: Milder Native Scale Weight ─────────────────
# Dropping the multiplier down from 54.12 to 12.0 to stabilize early trees
tuned_scale_weight = 12.0

model_params_alt = {
    "iterations"           : 1500,
    "learning_rate"        : 0.03,      
    "depth"                : 6,         # Dropped depth from 7 to 6 to fight overfitting
    "eval_metric"          : "AUC",
    "early_stopping_rounds": 100,
    "verbose"              : 200,
    "random_seed"          : 42,
    "loss_function"        : "Logloss",
    
    "scale_pos_weight"     : tuned_scale_weight,
    
    "l2_leaf_reg"          : 10,        # Increased from 5 to 10 to force generalization
    "border_count"         : 128,
    "cat_features"         : cat_features_for_cb
}

print("=" * 60)
print(f"🏃 RUNNING COMPROMISE EXPERIMENT | Scale Weight Set to: {tuned_scale_weight}")
print("=" * 60)

# ── Train the Alternate Model ───────────────────────────────────────────
model_alt = CatBoostClassifier(**model_params_alt)
model_alt.fit(
    X_train_pd, y_train_pd,
    eval_set=(X_val_pd, y_val_pd),
    use_best_model=True
)

# ── Threshold Search Grid Adjustment ────────────────────────────────────
print("\n🔍 Scanning intermediate probability bands for optimal threshold...")
val_probs_alt = model_alt.predict_proba(X_val_pd)[:, 1]

best_threshold_alt = 0.5
best_f1_alt = 0.0

# Because the weight is lower, the optimal threshold will drop down from 0.9995
for thresh in np.linspace(0.7000, 0.9990, 1000):
    preds = (val_probs_alt >= thresh).astype(int)
    score = f1_score(y_val_pd, preds, zero_division=0)
    if score > best_f1_alt:
        best_f1_alt = score
        best_threshold_alt = thresh

print(f"🎯 Alternate Threshold Located: {best_threshold_alt:.4f} (Val F1: {best_f1_alt:.4f})")

# ── Final Comparative Test Evaluation ───────────────────────────────────
test_probs_alt = model_alt.predict_proba(X_test_pd)[:, 1]
test_preds_alt = (test_probs_alt >= best_threshold_alt).astype(int)

print("\n" + "=" * 50)
print(f"  VERIFIED PERFORMANCE METRICS (TUNED OPTION)")
print("=" * 50)
print(f"  Final Test AUC       : {roc_auc_score(y_test_pd, test_probs_alt):.4f}")
print(f"  Final Test Precision : {precision_score(y_test_pd, test_preds_alt, zero_division=0):.4f}")
print(f"  Final Test Recall    : {recall_score(y_test_pd, test_preds_alt):.4f}")
print(f"  Final Test F1-Score  : {f1_score(y_test_pd, test_preds_alt, zero_division=0):.4f}")
print(f"  Operational Cutoff   : {best_threshold_alt:.4f}")
print("=" * 50)

🏃 RUNNING COMPROMISE EXPERIMENT | Scale Weight Set to: 12.0
0:	test: 0.6473189	best: 0.6473189 (0)	total: 425ms	remaining: 10m 37s
200:	test: 0.8238578	best: 0.8345228 (164)	total: 1m 41s	remaining: 10m 56s
400:	test: 0.8661549	best: 0.8661549 (400)	total: 3m 22s	remaining: 9m 16s
600:	test: 0.8917924	best: 0.8919967 (598)	total: 5m 3s	remaining: 7m 34s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.8929995023
bestIteration = 609

Shrink model to first 610 iterations.

🔍 Scanning intermediate probability bands for optimal threshold...
🎯 Alternate Threshold Located: 0.9990 (Val F1: 0.4328)

  VERIFIED PERFORMANCE METRICS (TUNED OPTION)
  Final Test AUC       : 0.8430
  Final Test Precision : 0.1891
  Final Test Recall    : 0.4785
  Final Test F1-Score  : 0.2711
  Operational Cutoff   : 0.9990


In [0]:
# (Scale = 5.0 & L2 = 15) >> better percision + F1-score (more than 1 out of every 3 alerts generated will be an actual confirmed fraud case)

cat_features_for_cb = [c for c in CAT_NAMES if c in feature_cols]

# ── FINAL TUNE CONFIGURATION: Milder Standardized Weight ────────────────
# Testing an even lighter penalty to see if it stabilizes future test precision
final_scale_weight = 5.0

model_params_final_tune = {
    "iterations"           : 1500,
    "learning_rate"        : 0.03,      
    "depth"                : 6,         
    "eval_metric"          : "AUC",
    "early_stopping_rounds": 100,
    "verbose"              : 200,
    "random_seed"          : 42,
    "loss_function"        : "Logloss",
    
    "scale_pos_weight"     : final_scale_weight,
    
    "l2_leaf_reg"          : 15,        # Slightly higher regularization to protect Test Set
    "border_count"         : 128,
    "cat_features"         : cat_features_for_cb
}

print("=" * 60)
print(f"🏃 RUNNING FINAL TUNE CHECK | Scale Weight Set to: {final_scale_weight}")
print("=" * 60)

model_ft = CatBoostClassifier(**model_params_final_tune)
model_ft.fit(X_train_pd, y_train_pd, eval_set=(X_val_pd, y_val_pd), use_best_model=True)

# Broadening search bounds to catch lower probability distributions safely
print("\n🔍 Scanning probability spectrum for optimal threshold...")
val_probs_ft = model_ft.predict_proba(X_val_pd)[:, 1]

best_threshold_ft = 0.5
best_f1_ft = 0.0

for thresh in np.linspace(0.5000, 0.9990, 1000):
    preds = (val_probs_ft >= thresh).astype(int)
    score = f1_score(y_val_pd, preds, zero_division=0)
    if score > best_f1_ft:
        best_f1_ft = score
        best_threshold_ft = thresh

print(f"🎯 Final Tune Threshold Located: {best_threshold_ft:.4f} (Val F1: {best_f1_ft:.4f})")

# ── Final Comparative Test Evaluation ───────────────────────────────────
test_probs_ft = model_ft.predict_proba(X_test_pd)[:, 1]
test_preds_ft = (test_probs_ft >= best_threshold_ft).astype(int)

print("\n" + "=" * 50)
print(f"  VERIFIED PERFORMANCE METRICS (FINAL TUNE CHECK)")
print("=" * 50)
print(f"  Final Test AUC       : {roc_auc_score(y_test_pd, test_probs_ft):.4f}")
print(f"  Final Test Precision : {precision_score(y_test_pd, test_preds_ft, zero_division=0):.4f}")
print(f"  Final Test Recall    : {recall_score(y_test_pd, test_preds_ft):.4f}")
print(f"  Final Test F1-Score  : {f1_score(y_test_pd, test_preds_ft, zero_division=0):.4f}")
print(f"  Operational Cutoff   : {best_threshold_ft:.4f}")
print("=" * 50)

🏃 RUNNING FINAL TUNE CHECK | Scale Weight Set to: 5.0
0:	test: 0.7821483	best: 0.7821483 (0)	total: 419ms	remaining: 10m 27s
200:	test: 0.8631306	best: 0.8650461 (163)	total: 1m 42s	remaining: 11m 2s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.8783464954
bestIteration = 257

Shrink model to first 258 iterations.

🔍 Scanning probability spectrum for optimal threshold...
🎯 Final Tune Threshold Located: 0.9980 (Val F1: 0.5091)

  VERIFIED PERFORMANCE METRICS (FINAL TUNE CHECK)
  Final Test AUC       : 0.8262
  Final Test Precision : 0.3442
  Final Test Recall    : 0.3821
  Final Test F1-Score  : 0.3622
  Operational Cutoff   : 0.9980


## MLflow

In [0]:

import mlflow.catboost
from catboost import CatBoostClassifier

# ── 1. Map Databricks Workspace Experiment Path ─────────────────────────
try:
    user_email = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
    experiment_path = f"/Users/{user_email}/fraud_detection"
except Exception:
    experiment_path = "/Shared/fraud_detection"

mlflow.set_experiment(experiment_path)

cat_features_for_cb = [c for c in CAT_NAMES if c in feature_cols]

# ── 2. Initialize the Champion Parameters ───────────────────────────────
production_params = {
    "iterations"           : 1500,
    "learning_rate"        : 0.03,      
    "depth"                : 6,         
    "eval_metric"          : "AUC",
    "early_stopping_rounds": 100,
    "verbose"              : 200,
    "random_seed"          : 42,
    "loss_function"        : "Logloss",
    
    # Locked-in tuned parameters
    "scale_pos_weight"     : 5.0,
    "l2_leaf_reg"          : 15,
    
    "border_count"         : 128,
    "cat_features"         : cat_features_for_cb
}

# ── 3. Execute Training & MLflow Logging Run ────────────────────────────
with mlflow.start_run(run_name="catboost_production") as run:

    mlflow.log_params({
        "imbalance_handling": "Tuned_Native_Scale_5.0",
        "feature_count": len(feature_cols),
        **{k: v for k, v in production_params.items() if k not in ["verbose", "cat_features"]},
    })

    print(" Training CatBoost model...")
    model = CatBoostClassifier(**production_params)
    model.fit(X_train_pd, y_train_pd, eval_set=(X_val_pd, y_val_pd), use_best_model=True)

    # ── 4. Apply the Production Threshold ────────────────────────────────
    # We lock in the exact operational threshold discovered during the final check
    production_threshold = 0.9980
    mlflow.log_param("optimal_threshold", production_threshold)

    # ── 5. Run Final Evaluation for Artifact Storage ─────────────────────
    test_probs = model.predict_proba(X_test_pd)[:, 1]
    test_preds = (test_probs >= production_threshold).astype(int)

    test_auc = roc_auc_score(y_test_pd, test_probs)
    test_prec = precision_score(y_test_pd, test_preds, zero_division=0)
    test_rec = recall_score(y_test_pd, test_preds)
    test_f1 = f1_score(y_test_pd, test_preds, zero_division=0)

    mlflow.log_metrics({
        "test_auc": test_auc,
        "test_precision": test_prec,
        "test_recall": test_rec,
        "test_f1": test_f1
    })

    print("\n" + "=" * 50)
    print(f" LOGGED SUCCESSFULLY TO MLFLOW")
    print("=" * 50)
    print(f"  Logged Test AUC       : {test_auc:.4f}")
    print(f"  Logged Test Precision : {test_prec:.4f}")
    print(f"  Logged Test Recall    : {test_rec:.4f}")
    print(f"  Logged Test F1-Score  : {test_f1:.4f}")
    print(f"  Operational Cutoff   : {production_threshold:.4f}")
    print("=" * 50)

    # ── 6. Log model artifact weights safely ─────────────────────────────
    try:
        mlflow.catboost.log_model(model, "catboost_fraud_champion_model")
    except Exception as e:
        print(f"⚠️ Note: Artifact upload skipped due to storage limit boundaries, metrics saved fine.")
        
    run_id = run.info.run_id
    print(f"\nUse this run ID for deployment: {run_id}")

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


 Training CatBoost model...
0:	test: 0.7821483	best: 0.7821483 (0)	total: 398ms	remaining: 9m 56s
200:	test: 0.8631306	best: 0.8650461 (163)	total: 1m 44s	remaining: 11m 12s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.8783464954
bestIteration = 257

Shrink model to first 258 iterations.


2026/07/05 00:51:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



 LOGGED SUCCESSFULLY TO MLFLOW
  Logged Test AUC       : 0.8262
  Logged Test Precision : 0.3437
  Logged Test Recall    : 0.3821
  Logged Test F1-Score  : 0.3619
  Operational Cutoff   : 0.9980


🔗 View Logged Model at: https://dbc-504ce144-2e40.cloud.databricks.com/ml/experiments/2599643044276425/models/m-e012471a4aa543779a9d2261a9cca347?o=7474650823001659
2026/07/05 00:51:53 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.14.0/ml/model/signatures.html for instructions on setting signature on models.



Use this run ID for deployment: 5d4d740e879344f9b3c344d7fffaf712


## Pickle Model + Metadata

In [0]:
model_artifact = {
    "model"          : model,                
    "feature_cols"   : feature_cols,         # Strict column ordering blueprint for Data Engineers
    "cat_features"   : CAT_NAMES_FOR_CB,     # Categorical index mapping markers
    "threshold"      : production_threshold, 
    "metrics"        : metrics,              
    "imbalance_ratio": raw_imbalance_ratio,  
    "mlflow_run_id"  : run_id,
}

PICKLE_PATH = "/Workspace/Users/<user>/fraud_Detection_ML/model_artifact.pkl"

dir_name = os.path.dirname(PICKLE_PATH)
if not os.path.exists(dir_name):
    os.makedirs(dir_name, exist_ok=True)
    print(f"Created target directory path: {dir_name}")

with open(PICKLE_PATH, "wb") as f:
    pickle.dump(model_artifact, f)

print(f"Model artifact successfully saved")
print("Verified  Metrics Footprint:")
print(json.dumps(model_artifact["metrics"], indent=2))

Model artifact successfully saved
Verified  Metrics Footprint:
{
  "AUC": 0.9278015769424595,
  "Precision": 0.020907129188550956,
  "Recall": 0.9422018348623853,
  "F1": 0.040906556201704775,
  "Threshold": 0.5
}


## SHAP Summary 

In [0]:
feature_importance = pd.DataFrame({
    "feature"   : feature_cols,
    "importance": model.get_feature_importance(),
}).sort_values("importance", ascending=False)

print("Top 20 features by importance:")
print(feature_importance.head(20).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(data=feature_importance.head(20), y="feature", x="importance",
            orient="h", ax=ax, palette="viridis")
ax.set_title("Feature Importance (CatBoost)")
plt.tight_layout()
plt.savefig("/tmp/feature_importance.png", dpi=150)
mlflow.log_artifact("/tmp/feature_importance.png")
plt.show()


Top 20 features by importance:
             feature  importance
                 amt   35.822279
                hour   27.388544
         avg_amt_24h    7.712963
              gender    7.506846
               month    6.919944
 time_since_last_txn    3.461237
           spend_24h    3.060615
            category    2.605509
                 zip    1.748651
            merchant    0.977095
       txn_count_24h    0.914470
        txn_count_1h    0.897617
unique_merchants_24h    0.609971
            city_pop    0.126295
                city    0.066729
                 job    0.059957
         day_of_week    0.055400
  haversine_distance    0.030603
               state    0.025765
            distance    0.009511


In [0]:
sample_idx = np.random.choice(len(X_test_pd), size=min(2000, len(X_test_pd)), replace=False)
X_shap_sample = X_test_pd.iloc[sample_idx]

shap_df, shap_values = compute_shap_reasons(model, X_shap_sample)

# Beeswarm / summary plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap_sample, show=False)
plt.tight_layout()
plt.savefig("/tmp/shap_summary.png", dpi=150)
mlflow.log_artifact("/tmp/shap_summary.png")
plt.show()
print("SHAP summary saved.")


SHAP summary saved.


## Stop Spark Session

In [0]:
spark.stop()